# Task 2 - ResNet-18 from Scratch on GTSRB

so the task is to implement ResNet-18 from the original paper:
'Deep Residual Learning for Image Recognition' by He et al. 2015

the key idea from the paper is adding shortcut connections (also called skip connections or residual connections) that let the input of a block bypass the conv layers and be added to the output directly.

mathematically: instead of learning H(x), the layers learn F(x) = H(x) - x
so the output is F(x) + x

at first i was like why does this matter but then i understood - in very deep networks gradients get smaller and smaller as they flow backwards (vanishing gradient problem). skip connections give the gradient a direct path backwards, so even in early layers the gradient doesnt vanish as much.

dataset: GTSRB german traffic sign recognition benchmark (43 classes)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

NUM_CLASSES = 43  # gtsrb has 43 traffic sign classes

## Dataset Loading

GTSRB comes with a Train and Test folder. each class has its own subfolder inside Train.

In [ ]:
DATA_PATH = './data/GTSRB'
TRAIN_PATH = os.path.join(DATA_PATH, 'Train')
TEST_PATH = os.path.join(DATA_PATH, 'Test')

# check what classes we have
classes = sorted(os.listdir(TRAIN_PATH))
print('number of classes:', len(classes))
print('first few classes:', classes[:5])

In [ ]:
class GTSRBDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        # loop through each class folder and collect image paths + labels
        for label_idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if fname.endswith('.png') or fname.endswith('.ppm'):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        int(label_idx)
                    ))
        
        print(f'total samples loaded: {len(self.samples)}')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

In [ ]:
# resnet takes 224x224 inputs according to the paper
# i also add normalization using imagenet mean and std because thats what the paper uses
# (even though we're training from scratch the normalization still helps)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# load all train data and split 80/20
full_train = GTSRBDataset(TRAIN_PATH, transform=None)
n_train = int(0.8 * len(full_train))
n_val = len(full_train) - n_train

# manual split
all_samples = full_train.samples
np.random.seed(42)
np.random.shuffle(all_samples)

train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:]

# create two separate dataset objects with different transforms
class SplitDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = SplitDataset(train_samples, train_transform)
val_dataset = SplitDataset(val_samples, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print('train batches:', len(train_loader))
print('val batches:', len(val_loader))

## ResNet-18 Architecture

from the paper, ResNet-18 has this structure:
- initial conv layer: 7x7, 64 filters, stride 2
- max pool
- 4 stages of residual blocks: [2,2,2,2] blocks each
- global average pooling
- fully connected layer to number of classes

### The Residual Block

each residual block (BasicBlock for ResNet-18) has:
- conv 3x3
- batch norm
- relu
- conv 3x3
- batch norm
- then ADD the input (shortcut)
- relu after the add

the tricky part is the shortcut connection when the dimensions change (when stride=2 or when channels increase). in that case we need a 1x1 conv on the shortcut to match dimensions. this is called a "projection shortcut" in the paper.

mathematically:
- output = ReLU(F(x) + x)  -- when dimensions match
- output = ReLU(F(x) + W_s * x)  -- when dimensions change, W_s is the 1x1 conv

### Why does this help with vanishing gradients?

during backprop, gradient flows through: d(loss)/d(x) = d(loss)/d(output) * d(output)/d(x)

with residual connection: d(F(x)+x)/d(x) = dF/dx + 1

the +1 term means even if dF/dx becomes very small (vanishes), the gradient still has a constant flow of 1 through the shortcut. so gradients dont vanish as easily in early layers.

In [ ]:
class BasicBlock(nn.Module):
    """
    basic residual block for resnet-18 and resnet-34
    two 3x3 convolutions with a shortcut connection
    """
    
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        
        # first conv layer
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # second conv layer - stride is always 1 here
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # shortcut/skip connection
        # if the input and output dimensions are different, we need to project the input
        # using 1x1 conv to match dimensions so we can add them
        self.shortcut = nn.Identity()  # default: just pass through
        
        if stride != 1 or in_channels != out_channels:
            # need to match dimensions for the addition
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.shortcut(x)  # save input for later addition
        
        out = self.relu(self.bn1(self.conv1(x)))  # first conv
        out = self.bn2(self.conv2(out))  # second conv (no relu yet!)
        
        out = out + identity  # this is the key residual connection: F(x) + x
        out = self.relu(out)  # relu AFTER the addition
        
        return out

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=43):
        super(ResNet18, self).__init__()
        
        # first layer: 7x7 conv, 64 filters, stride 2 (from paper table 1)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # the 4 stages with [2,2,2,2] blocks each - this is what makes it ResNet-18
        # (18 = 1 initial + 2*4*2 conv layers inside blocks + 1 fc = 1 + 16 + 1 = 18)
        # stage 1: 64 channels, no downsample
        self.layer1 = self._make_stage(64, 64, num_blocks=2, stride=1)
        
        # stage 2: 128 channels, downsample with stride=2
        self.layer2 = self._make_stage(64, 128, num_blocks=2, stride=2)
        
        # stage 3: 256 channels, downsample
        self.layer3 = self._make_stage(128, 256, num_blocks=2, stride=2)
        
        # stage 4: 512 channels, downsample
        self.layer4 = self._make_stage(256, 512, num_blocks=2, stride=2)
        
        # global average pooling then flatten then fc
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))  # output is 1x1 regardless of input size
        self.fc = nn.Linear(512, num_classes)
        
        # weight initialization like the paper does
        self._init_weights()
    
    def _make_stage(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        # first block handles downsampling (if any)
        layers.append(BasicBlock(in_channels, out_channels, stride=stride))
        # remaining blocks have same dimensions
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # kaiming (he) initialization for conv layers
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

In [ ]:
# verify architecture with dummy input
model = ResNet18(num_classes=43).to(device)

dummy = torch.zeros(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print('output shape:', out.shape)  # should be [2, 43]

# count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'total parameters: {total_params:,}')
# resnet18 should have around 11 million params

## Training

for classification i use CrossEntropyLoss which combines softmax + log + negative log likelihood. its the standard choice for multi-class classification.

i also track accuracy here which is more interpretable than loss for classification.

In [ ]:
model = ResNet18(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()

# SGD with momentum like in the original resnet paper
# they use lr=0.1 but since our dataset is smaller i start with 0.01
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# cosine annealing to reduce lr over time
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

NUM_EPOCHS = 30

train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # -- train --
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    train_loss = total_loss / len(train_loader)
    train_acc = correct / total
    
    # -- validate --
    model.eval()
    val_loss_total = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            val_loss_total += loss.item()
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_loss = val_loss_total / len(val_loader)
    val_acc = val_correct / val_total
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step()
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'resnet18_best.pth')
    
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.3f} | Acc: {train_acc:.3f} | Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}')

print(f'\nbest val accuracy: {best_val_acc:.4f}')

## Results and Visualizations

In [ ]:
# loss and accuracy curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss', color='blue')
ax1.plot(val_losses, label='Val Loss', color='orange')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Train Acc', color='blue')
ax2.plot(val_accs, label='Val Acc', color='orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('resnet18_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# confusion matrix - shows which classes the model confuses with each other
# for 43 classes its going to be a big matrix

model.load_state_dict(torch.load('resnet18_best.pth', map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        outputs = model(imgs.to(device))
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(15, 13))
sns.heatmap(cm, cmap='Blues', fmt='d', 
            xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix - ResNet18 on GTSRB')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('resnet18_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

# overall accuracy from confusion matrix
acc = np.diag(cm).sum() / cm.sum()
print(f'Overall accuracy: {acc:.4f}')

In [ ]:
# show some predictions visually
# traffic sign class names (from gtsrb documentation)
class_names = [
    'Speed 20', 'Speed 30', 'Speed 50', 'Speed 60', 'Speed 70',
    'Speed 80', 'End 80', 'Speed 100', 'Speed 120', 'No Passing',
    'No Passing >3.5t', 'Priority', 'Priority Road', 'Yield', 'Stop',
    'No Vehicles', 'No >3.5t', 'No Entry', 'General Caution', 'Dangerous Left',
    'Dangerous Right', 'Double Curve', 'Bumpy Road', 'Slippery Road', 'Road Narrows',
    'Road Work', 'Traffic Signals', 'Pedestrians', 'Children', 'Bicycles',
    'Ice/Snow', 'Wild Animals', 'End Restrictions', 'Turn Right', 'Turn Left',
    'Go Straight', 'Straight or Right', 'Straight or Left', 'Keep Right', 'Keep Left',
    'Roundabout', 'End No Passing', 'End No Passing >3.5t'
]

# pick a few samples
model.eval()
sample_imgs, sample_labels = next(iter(val_loader))

with torch.no_grad():
    outputs = model(sample_imgs.to(device))
    preds = outputs.argmax(dim=1).cpu().numpy()

sample_labels = sample_labels.numpy()

# un-normalize the images for display
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for i in range(10):
    img = sample_imgs[i].permute(1, 2, 0).numpy()
    img = img * std + mean  # denormalize
    img = np.clip(img, 0, 1)
    
    true_class = class_names[sample_labels[i]]
    pred_class = class_names[preds[i]]
    correct = sample_labels[i] == preds[i]
    
    axes[i].imshow(img)
    color = 'green' if correct else 'red'
    axes[i].set_title(f'True: {true_class}\nPred: {pred_class}', 
                      color=color, fontsize=7)
    axes[i].axis('off')

plt.suptitle('Sample Predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.savefig('resnet18_sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# per-class accuracy to see which signs are hardest
per_class_acc = np.diag(cm) / cm.sum(axis=1)

plt.figure(figsize=(14, 5))
bars = plt.bar(range(43), per_class_acc, color='steelblue')

# highlight lowest accuracy classes
bottom5_idx = np.argsort(per_class_acc)[:5]
for idx in bottom5_idx:
    bars[idx].set_color('red')

plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy (red = worst 5)')
plt.xticks(range(43), rotation=90)
plt.grid(axis='y')
plt.tight_layout()
plt.savefig('resnet18_per_class_acc.png', dpi=100, bbox_inches='tight')
plt.show()

print('Worst 5 classes:')
for idx in bottom5_idx:
    print(f'  Class {idx} ({class_names[idx]}): {per_class_acc[idx]:.3f}')

## Summary

### What I learned building ResNet-18:

**The residual connection:**
mathematically output = F(x) + x means during backprop the gradient is dL/dx = dL/d(output) * (dF/dx + 1). that constant 1 ensures gradient always has a clear path backwards even if dF/dx is tiny. plain networks dont have this, so in very deep networks (like 50+ layers) gradients often become near-zero in early layers and those layers stop learning. 

intuitively: instead of learning the full transformation, the block only needs to learn the *residual* (the difference from identity). if the best thing to do is nothing, the block can just push F(x) to zero and the output will just be x. this makes optimization easier.

**The projection shortcut:**
when channels change (e.g., 64->128) or spatial size changes (stride=2), we cant just add x directly because shapes dont match. so we use a 1x1 conv to project x to the right shape. i spent a while debugging this part.

**BatchNorm placement:**
the paper puts BatchNorm after conv and before relu. the activation is applied AFTER the residual addition, not inside the block. this is important - i got this wrong the first time.

ResNet-18 should outperform my Task 0 custom CNN by a decent margin on GTSRB because the residual connections allow training a deeper network effectively.

yes , I used ai in many plaaces,one because i was comfortable with tf but here i have to strictly implement pytorch, everything seemed difficult moving from tf. 
Two, i dont know plotting and visualizing very well, as i started my ml very recently
third, i used ai to write this report, what ever i did along with ai is summarized.